# 🏁 Неделя 4 — Финал: Выводы и результаты
**Датасет:** Student Performance (UCI)  
**Задача:** Предсказать итоговую оценку студента (G3)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import urllib.request, zipfile, os
os.makedirs('../data/raw', exist_ok=True)
if not os.path.exists('../data/raw/student-mat.csv'):
    urllib.request.urlretrieve(
        'https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip',
        '../data/raw/student.zip'
    )
    with zipfile.ZipFile('../data/raw/student.zip') as z:
        z.extractall('../data/raw')
    os.remove('../data/raw/student.zip')

df = pd.read_csv('../data/raw/student-mat.csv', sep=';')

cat_cols = df.select_dtypes(include='object').columns
le = LabelEncoder()
df_enc = df.copy()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df[col])

X = df_enc.drop('G3', axis=1)
y = df_enc['G3']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred = gb.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print('Лучшая модель: Gradient Boosting')
print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²:   {r2:.3f}')

## 1. Предсказания vs Реальные оценки

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white')
plt.plot([0, 20], [0, 20], 'r--', linewidth=2, label='Идеальное предсказание')
plt.title('Предсказания модели vs Реальные оценки', fontsize=14)
plt.xlabel('Реальная оценка G3')
plt.ylabel('Предсказанная оценка')
plt.legend()
plt.tight_layout()
plt.show()

## 2. Важность признаков — финальный анализ

In [ ]:
importances = pd.Series(gb.feature_importances_, index=X.columns)
top10 = importances.sort_values(ascending=True).tail(10)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v > 0.1 else 'steelblue' for v in top10.values]
top10.plot(kind='barh', color=colors, edgecolor='white')
plt.title('Топ-10 важных признаков (Gradient Boosting)', fontsize=14)
plt.xlabel('Важность признака')
plt.ylabel('Признак')
plt.tight_layout()
plt.show()

## 3. Выводы по проекту

### Что узнал:
1. **G1 и G2** (оценки за предыдущие периоды) — самые важные признаки для предсказания итоговой оценки. Это значит, что успеваемость студента стабильна на протяжении года.
2. **Количество провалов (failures)** сильно снижает итоговую оценку — студенты с провалами в прошлом учатся хуже.
3. **Время на учёбу и образование родителей** также влияют на результат, но слабее.

### Какие признаки оказались важными:
- 🥇 G2 — оценка за 2-й период (важность ~0.65)
- 🥈 G1 — оценка за 1-й период (важность ~0.15)
- 🥉 failures — количество провалов (важность ~0.05)
- absences, studytime, Medu — менее важные, но полезные признаки

### Что можно улучшить:
1. **Добавить новые признаки** — например, время в дороге до школы, качество отношений с учителями
2. **Настроить гиперпараметры** моделей (GridSearchCV) для повышения точности
3. **Попробовать XGBoost или LightGBM** — более продвинутые алгоритмы бустинга
4. **Объединить датасеты** по математике и португальскому языку для большего объёма данных
5. **Убрать G1 и G2 из признаков** — интереснее предсказывать оценку без промежуточных результатов

## 4. Итоговые метрики

| Модель | MAE | RMSE | R² |
|---|---|---|---|
| Линейная регрессия | ~1.5 | ~2.1 | ~0.82 |
| Random Forest | ~1.2 | ~1.8 | ~0.88 |
| **Gradient Boosting** | **~1.1** | **~1.6** | **~0.91** |

**Лучшая модель: Gradient Boosting** — R² около 0.91, что означает что модель объясняет 91% вариации в оценках студентов.